# **Build a Speed Dating Match Prediction Agent w/ LangGraph**


In [22]:
%%capture
%pip install langchain
%pip install langchain-openai
%pip install langgraph
%pip install openai
%pip install numpy
%pip install pandas
%pip install scikit-learn
%pip install scipy
%pip install xgboost
%pip install tqdm
%pip install joblib
%pip install requests
%pip install PyYAML

In [23]:
import pandas as pd
import numpy as np
import io
import os
from typing import Annotated, List, Dict, Any, Union
from typing_extensions import TypedDict
from operator import add

from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.metrics import accuracy_score, roc_auc_score
import xgboost as xgb

Fetching the pseed dating dataset by running the following code.


In [24]:
!wget https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/sqiJ_CW9x_k2T6C2KgPf6Q/speeddating.csv

'wget' is not recognized as an internal or external command,
operable program or batch file.


# Introduction


## Why Use an Agent for ML Workflows?


Traditional ML pipelines require you to manually:
- Remember the correct order of operations
- Pass data between functions
- Handle errors at each step
- Track intermediate results

An **agentic workflow** automates this by:
- Letting the AI decide the sequence of operations
- Managing state automatically
- Providing transparency into the reasoning process
- Handling complex multistep tasks with a single instruction



# Understanding the Speed Dating Dataset


The speed dating dataset is a public dataset collected by [Ulrik Thyge Pedersen](https://www.kaggle.com/datasets/ulrikthygepedersen/speed-dating/data). It contains information from real speed dating events where participants rated each other on various attributes. Each row represents one person's ratings of another person during a 4-minute date.

**Key Features in the Dataset:**
- **Demographic info**: age, race, field of study
- **Attribute ratings**: attractiveness, sincerity, intelligence, fun, ambition, shared interests (rated 1 to 10)
- **Decision variables**: `decision` (did this person want to see the other again?), `decision_o` (did the partner want to see them again?)
- **Match outcome**: `match` (1 if both said yes, 0 otherwise)

**The Challenge:**
The `decision` and `decision_o` columns are **data leakage**—if we know both individual decisions, we automatically know if there's a match. For a realistic probability prediction, we need to remove these columns and predict matches based only on ratings and demographics.

Additionally, the dataset has:
- Byte string encodings (e.g., `b'female'` instead of `'female'`)
- Missing values that need imputation
- Categorical variables that need encoding

Our agent will handle all of these automatically!



## Exploratory Data Analysis

Let's take a quick look at the dataset to understand what the agent will be working with. We'll load the data and check its shape, basic info, and the class distribution for our target variable `match`.

In [25]:
import pandas as pd

# Load the dataset for EDA
df_eda = pd.read_csv('speeddating.csv', low_memory=False)

# Display the shape and the first few rows
print(f"Dataset Shape: {df_eda.shape}\n")
display(df_eda.head())

Dataset Shape: (8378, 123)



,has_null,wave,gender,age,age_o,d_age,d_d_age,race,race_o,samerace,...,d_expected_num_interested_in_me,d_expected_num_matches,like,guess_prob_liked,d_like,d_guess_prob_liked,met,decision,decision_o,match
0,b'',1.0,b'female',21.0,27.0,6.0,b'[4-6]',b'Asian/Pacific Islander/Asian-American',b'European/Caucasian-American',b'0',...,b'[0-3]',b'[3-5]',7.0,6.0,b'[6-8]',b'[5-6]',0.0,b'1',b'0',b'0'
1,b'',1.0,b'female',21.0,22.0,1.0,b'[0-1]',b'Asian/Pacific Islander/Asian-American',b'European/Caucasian-American',b'0',...,b'[0-3]',b'[3-5]',7.0,5.0,b'[6-8]',b'[5-6]',1.0,b'1',b'0',b'0'
2,b'',1.0,b'female',21.0,22.0,1.0,b'[0-1]',b'Asian/Pacific Islander/Asian-American',b'Asian/Pacific Islander/Asian-American',b'1',...,b'[0-3]',b'[3-5]',7.0,NaN,b'[6-8]',b'[0-4]',1.0,b'1',b'1',b'1'
3,b'',1.0,b'female',21.0,23.0,2.0,b'[2-3]',b'Asian/Pacific Islander/Asian-American',b'European/Caucasian-American',b'0',...,b'[0-3]',b'[3-5]',7.0,6.0,b'[6-8]',b'[5-6]',0.0,b'1',b'1',b'1'
4,b'',1.0,b'female',21.0,24.0,3.0,b'[2-3]',b'Asian/Pacific Islander/Asian-American',b'Latino/Hispanic American',b'0',...,b'[0-3]',b'[3-5]',6.0,6.0,b'[6-8]',b'[5-6]',0.0,b'1',b'1',b'1'


In [26]:
# Check the data types and look out for missing values or byte strings
df_eda.info()

<class 'pandas.DataFrame'>
RangeIndex: 8378 entries, 0 to 8377
Columns: 123 entries, has_null to match
dtypes: float64(59), str(64)
memory usage: 12.5 MB


In [27]:
# Check the target variable distribution
# 0 = No Match, 1 = Match
match_dist = df_eda['match'].value_counts(normalize=True) * 100
print("Match Distribution (Percentage):")
print(match_dist)

print("\nAs we can see, this dataset is imbalanced. Most dates do not result in a match!")

Match Distribution (Percentage):
match
b'0'    83.528288
b'1'    16.471712
Name: proportion, dtype: float64

As we can see, this dataset is imbalanced. Most dates do not result in a match!


In [28]:
# Display basic summary statistics for the numerical features
display(df_eda.describe())

,wave,age,age_o,d_age,importance_same_race,importance_same_religion,pref_o_attractive,pref_o_sincere,pref_o_intelligence,pref_o_funny,...,music,shopping,yoga,interests_correlate,expected_happy_with_sd_people,expected_num_interested_in_me,expected_num_matches,like,guess_prob_liked,met
count,8378.000000,8283.000000,8274.000000,8378.000000,8299.000000,8299.000000,8289.000000,8289.000000,8289.000000,8280.000000,...,8299.000000,8299.000000,8299.000000,8220.000000,8277.000000,1800.000000,7205.000000,8138.000000,8069.000000,8003.000000
mean,11.350919,26.358928,26.364999,4.185605,3.784793,3.651645,22.495347,17.396867,20.270759,17.459714,...,7.851066,5.631281,4.339197,0.196010,5.534131,5.570556,3.207814,6.134087,5.207523,0.049856
std,5.995903,3.566763,3.563648,4.596171,2.845708,2.805237,12.569802,7.044003,6.782895,6.085526,...,1.791827,2.608913,2.717612,0.303539,1.734059,4.762569,2.444813,1.841285,2.129565,0.282168
min,1.000000,18.000000,18.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,...,1.000000,1.000000,0.000000,-0.830000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,7.000000,24.000000,24.000000,1.000000,1.000000,1.000000,15.000000,15.000000,17.390000,15.000000,...,7.000000,4.000000,2.000000,-0.020000,5.000000,2.000000,2.000000,5.000000,4.000000,0.000000
50%,11.000000,26.000000,26.000000,3.000000,3.000000,3.000000,20.000000,18.370000,20.000000,18.000000,...,8.000000,6.000000,4.000000,0.210000,6.000000,4.000000,3.000000,6.000000,5.000000,0.000000
75%,15.000000,28.000000,28.000000,5.000000,6.000000,6.000000,25.000000,20.000000,23.810000,20.000000,...,9.000000,8.000000,7.000000,0.430000,7.000000,8.000000,4.000000,7.000000,7.000000,0.000000
max,21.000000,55.000000,55.000000,37.000000,10.000000,10.000000,100.000000,60.000000,50.000000,50.000000,...,10.000000,10.000000,10.000000,0.910000,10.000000,20.000000,18.000000,10.000000,10.000000,8.000000


# Building The Agent


## Step 1: Define the Tools (The "Actions")


Tools are the actions our agent can take. Each tool is a Python function decorated with `@tool` that performs a specific task. Let's define three tools for our ML pipeline:


### Tool 1: Clean Speed Dating Data


In [ ]:
@tool
def clean_speed_dating_data(file_name: str):
    """Cleans speeddating.csv, handles bytes-strings, and removes leakage columns."""
    df = pd.read_csv(file_name)
    
    # 1. Drop Leakage: 'decision' and 'decision_o' are the individual votes.
    # If we know these, the match is 100% certain. For probability, we drop them.
    df = df.drop(columns=['has_null', 'decision', 'decision_o'], errors='ignore')

    # 2. Clean byte-strings (e.g., b'female' -> female)
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].astype(str).str.replace("b'", "").str.replace("'", "")
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])

    # 3. Impute missing values
    imputer = SimpleImputer(strategy='median')
    df_clean = pd.DataFrame(imputer.fit_transform(df), columns=df.columns)
    
    df_clean.to_csv('cleaned_data.csv', index=False)
    return f"Data cleaned. Rows: {len(df_clean)}. Saved to 'cleaned_data.csv'."

**What This Tool Does:**
- Removes data leakage columns (`decision`, `decision_o`)
- Converts byte strings to regular strings and encodes categorical variables
- Fills missing values with median values
- Saves the cleaned data for the next step


### Tool 2: Select Top Features


In [ ]:
@tool
def select_top_features(n_features: int):
    """Uses Recursive Feature Elimination to find the best features for match probability."""
    df = pd.read_csv('cleaned_data.csv')
    X = df.drop(columns=['match'])
    y = df['match']
    
    selector = RFE(RandomForestClassifier(n_estimators=50), n_features_to_select=n_features)
    selector.fit(X, y)
    
    features = X.columns[selector.support_].tolist()
    return {"selected_features": features}

**What This Tool Does:**
- Uses Recursive Feature Elimination (RFE) to identify the most predictive features
- Trains a Random Forest to rank feature importance
- Returns the top N features that best predict matches


### Tool 3: Train Probability Model


In [ ]:
@tool
def train_probability_model(features: List[str]):
    """Trains XGBoost and returns the ROC AUC (probability quality score)."""
    df = pd.read_csv('cleaned_data.csv')
    X = df[features]
    y = df['match']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)
    
    model = xgb.XGBClassifier(n_estimators=100, eval_metric='logloss')
    model.fit(X_train, y_train)
    
    # Predict Probability
    y_prob = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    
    return f"Model trained on {features}. ROC AUC Score: {auc:.4f}. The probability predictions are reliable."

**What This Tool Does:**
- Trains an XGBoost classifier on the selected features
- Predicts match probabilities (not just yes/no classifications)
- Evaluates using ROC AUC, which measures how well the model ranks matches vs. non matches


## Step 2: Create the Agent State


The state is the "memory" that flows through the workflow. For our agent, we need to track the conversation messages.


In [ ]:
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add]

**Why Use `Annotated[List[BaseMessage], add]`?**
- The `add` operator tells LangGraph to **append** new messages to the list rather than replacing it
- This creates a full conversation history that the agent can reference
  



## Step 3: Build the Agent Reasoning Node


The reasoning node is where the agent thinks about what to do next.


In [ ]:
# --- 2. THE GRAPH (The "Reasoning" Loop) ---

class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], add]

tools = [clean_speed_dating_data, select_top_features, train_probability_model]
openai_api_key = os.environ.get('OPENAI_API_KEY')
llm = ChatOpenAI(model="gpt-4o-mini", 
                 api_key=openai_api_key,
                 temperature=0).bind_tools(tools)

def agent_reasoning(state: AgentState):
    system_prompt = SystemMessage(content=(
        "You are a Match Prediction Expert. You must use the tools sequentially. "
        "First, clean the data. Then, select the best features. "
        "Finally, train the model and report the probability quality (ROC AUC)."
    ))
    response = llm.invoke([system_prompt] + state["messages"])
    return {"messages": [response]}

**What This Does:**
- Takes the current state (conversation history)
- Adds a system prompt that guides the agent's behavior
- Calls the LLM to decide what to do next
- Returns the LLM's response (which may include tool calls)



## Step 4: Construct the LangGraph Workflow


Now we connect everything together into a graph.


In [ ]:
# Define Graph Logic
workflow = StateGraph(AgentState)
workflow.add_node("scientist", agent_reasoning)
workflow.add_node("tools", ToolNode(tools))
workflow.set_entry_point("scientist")

# ReAct Loop logic
def router(state):
    if state["messages"][-1].tool_calls:
        return "call_tool"
    return "end"

workflow.add_conditional_edges("scientist", router, {"call_tool": "tools", "end": END})
workflow.add_edge("tools", "scientist")

app = workflow.compile()

**How This Works:**
1. The `scientist` node generates a response
2. The `router` checks if the response contains tool calls
3. If yes → go to the `tools` node
4. If no → the agent is done, go to `END`
5. After tools execute → return to `scientist` to process results

This creates a **loop** where the agent can use multiple tools in sequence!




#  Running the Agent


Now let's run our agent with a complex instruction:


In [ ]:
query = "Clean 'speeddating.csv', select top 10 features, and predict match probability."

for output in app.stream({"messages": [HumanMessage(content=query)]}, stream_mode="updates"):
    for node_name, state_update in output.items():
        # 1. Capture the Agent's Thought/Reasoning
        if node_name == "scientist":
            message = state_update["messages"][-1]
            
            # If the agent is calling tools
            if message.tool_calls:
                print(f"\n🤔 THOUGHT:")
                # Sometimes content is empty when calling tools, so we describe the intent
                print(f"I need to use tools to process the data. Calling: {[t['name'] for t in message.tool_calls]}")
                
                for tool_call in message.tool_calls:
                    print(f"🛠️  ACTION: {tool_call['name']}({tool_call['args']})")
            
            # If it's the final response (no more tool calls)
            else:
                print(f"\n✅ FINAL ANALYSIS:")
                print("-" * 20)
                print(message.content)
                print("-" * 20)

        # 2. Capture the Tool's Output (Observation)
        elif node_name == "tools":
            # The tool node returns a list of ToolMessages
            for tool_msg in state_update["messages"]:
                print(f"\n👁️  OBSERVATION:")
                # We limit long outputs like CSV summaries for readability
                print(f"{str(tool_msg.content)[:500]}...") 

print("\n" + "="*50)
print("🏁 Workflow Complete.")

# Understanding the Output


When you run the agent, you'll see a conversation unfold:

**Expected Output Flow:**

🤔 THOUGHT:
I need to use tools to process the data. Calling: ['clean_speed_dating_data']
🛠️  ACTION: clean_speed_dating_data({'file_name': 'speeddating.csv'})

👁️  OBSERVATION:
Data cleaned. Rows: 8378. Saved to 'cleaned_data.csv'....

🤔 THOUGHT:
I need to use tools to process the data. Calling: ['select_top_features']
🛠️  ACTION: select_top_features({'n_features': 10})

👁️  OBSERVATION:
{"selected_features": ["field", "pref_o_attractive", "attractive_o", "funny_o", "shared_interests_o", "attractive_important", "attractive_partner", "interests_correlate", "like", "guess_prob_liked"]}...

🤔 THOUGHT:
I need to use tools to process the data. Calling: ['train_probability_model']
🛠️  ACTION: train_probability_model({'features': ['field', 'pref_o_attractive', 'attractive_o', 'funny_o', 'shared_interests_o', 'attractive_important', 'attractive_partner', 'interests_correlate', 'like', 'guess_prob_liked']})

👁️  OBSERVATION:
Model trained on ['field', 'pref_o_attractive', 'attractive_o', 'funny_o', 'shared_interests_o', 'attractive_important', 'attractive_partner', 'interests_correlate', 'like', 'guess_prob_liked']. ROC AUC Score: 0.8341. The probability predictions are reliable....

✅ FINAL ANALYSIS:

The data from 'speeddating.csv' has been cleaned, and the top 10 features selected for predicting match probability are:

1. field
2. pref_o_attractive
3. attractive_o
4. funny_o
5. shared_interests_o
6. attractive_important
7. attractive_partner
8. interests_correlate
9. like
10. guess_prob_liked

The model has been trained using these features, and the ROC AUC score is **0.8341**, indicating that the probability predictions are reliable.


🏁 Workflow Complete.


**What Just Happened?**
1. The agent decided it needed to clean data first
2. It called the cleaning tool and received confirmation
3. It then decided to select features
4. It received the top 10 features
5. It trained a model using those features
6. It received the ROC AUC score
7. Finally, it synthesized all results into a coherent summary

All of this happened **autonomously** from a single instruction!


# Conclusion

What is done:

- **Created three specialized tools** for data cleaning, feature selection, and model training
- **Built a ReAct agent** that autonomously decides which tools to use and when
- **Designed a LangGraph workflow** that manages the entire ML pipeline with a single instruction
- **Learned about state management** and how agents maintain context across multiple tool calls
- **Evaluated match predictions** using ROC AUC scores for probability based predictions
- **Handled real world data challenges** like byte strings, missing values, and data leakage

This project demonstrated how agentic workflows can transform complex, multistep data science tasks into simple, natural language instructions. The same pattern can be extended to:
- Hyperparameter tuning experiments
- Multi model comparisons
- Feature engineering pipelines
- Automated reporting and visualization

